In [ ]:
import numpy as np
import pandas as pd

from imblearn.combine import SMOTEENN
from imblearn.over_sampling import BorderlineSMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from models.MLPipeline import *
from utils import ASSETS_DIR

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import KNNImputer
from sklearn.ensemble import RandomForestClassifier


In [ ]:
df = pd.read_parquet(ASSETS_DIR / 'final_df.parquet')

In [ ]:


# 1. Separiamo la matrice delle Feature (X) dal Target (y)
X = df.drop(columns=['TARGET'])
X.drop(columns=['LBDEVAL'], inplace=True, errors='ignore')
y = df['TARGET']

# 2. TRAIN-TEST SPLIT (80% Train, 20% Test)
# stratify=y è fondamentale per mantenere le stesse percentuali di malati nei due set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Buchi (NaN) iniziali in X_train: {X_train.isna().sum().sum()}")

# 3. CONFIGURAZIONE DEL KNN IMPUTER

imputer = KNNImputer(n_neighbors=7, weights='distance')

# 4. ADDESTRAMENTO E TRASFORMAZIONE
# Il modello "impara" le distribuzioni SOLO da X_train
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)

# Il modello applica quanto imparato su X_test (senza barare)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

# 5. ARROTONDAMENTO PER DATI CLINICI DISCRETI
# Riportiamo le medie del KNN a numeri interi (es. 0.66 diventa 1.0)
X_train_imp = X_train_imp.round()
X_test_imp = X_test_imp.round()

print(f"Buchi (NaN) finali in X_train: {X_train_imp.isna().sum().sum()}")
print(f"Buchi (NaN) finali in X_test: {X_test_imp.isna().sum().sum()}")

In [ ]:
scaler = MinMaxScaler()

x_train_scaled = scaler.fit_transform(X_train_imp)
x_test_scaled = scaler.transform(X_test_imp)

In [ ]:
rf_model = RandomForestClassifier(
        n_estimators=100, 
        random_state=42,
        class_weight='balanced',
        n_jobs=-1,
    )

smote_enn = SMOTEENN(random_state=42)
smote_nc = BorderlineSMOTE(random_state=42)
pipeline_rf = ImbPipeline(steps=[
    ('smoteenn', smote_nc),
    ('Classifier', rf_model)
])

y_true_bin, y_proba = train_model_evaluate(x_train_scaled, y_train, pipeline_rf)

In [ ]:
generate_predictions_and_cm(x_train_scaled, y_train, pipeline_rf)
plot_reliability_diagram(y_train, y_proba[:, 2], title='Reliability Diagram - Random Forest (Train Set)')